![clothing_classification](clothing_classification.png)


Fashion Forward is a new AI-based e-commerce clothing retailer.
They want to use image classification to automatically categorize new product listings, making it easier for customers to find what they're looking for. It will also assist in inventory management by quickly sorting items.

As a data scientist tasked with implementing a garment classifier, your primary objective is to develop a machine learning model capable of accurately categorizing images of clothing items into distinct garment types such as shirts, trousers, shoes, etc.

In [15]:
# Run the cells below first

In [16]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics import Accuracy, Precision, Recall

In [17]:
# Load datasets
from torchvision import datasets
import torchvision.transforms as transforms

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

In [18]:
# Start coding here

# DataLoaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

# ── CNN Model ─────────────────────────────────────────────────
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28x28 → 28x28
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 28x28 → 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 14x14 → 14x14
            nn.ReLU(),
            nn.MaxPool2d(2, 2)                            # 14x14 → 7x7
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# ── Training ──────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FashionCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 2
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

# ── Predictions on test set ───────────────────────────────────
model.eval()
predictions = []
all_labels  = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().tolist()
        predictions.extend(preds)
        all_labels.extend(labels.tolist())

# ── Metrics ───────────────────────────────────────────────────
num_classes = 10
all_labels_tensor  = torch.tensor(all_labels)
predictions_tensor = torch.tensor(predictions)

acc_metric  = Accuracy(task='multiclass', num_classes=num_classes)
prec_metric = Precision(task='multiclass', num_classes=num_classes, average=None)
rec_metric  = Recall(task='multiclass', num_classes=num_classes, average=None)

accuracy  = acc_metric(predictions_tensor, all_labels_tensor).item()
precision = prec_metric(predictions_tensor, all_labels_tensor).tolist()
recall    = rec_metric(predictions_tensor, all_labels_tensor).tolist()

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision per class: {[round(p, 4) for p in precision]}")
print(f"Recall per class:    {[round(r, 4) for r in recall]}")

Epoch 1/2 - Loss: 0.4746
Epoch 2/2 - Loss: 0.3029

Accuracy: 0.8933
Precision per class: [0.8156, 0.9677, 0.8405, 0.9123, 0.8028, 0.9733, 0.7252, 0.9259, 0.9818, 0.9883]
Recall per class:    [0.858, 0.988, 0.817, 0.874, 0.851, 0.986, 0.681, 0.974, 0.972, 0.932]
